In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns",None)

In [2]:
df=pd.read_excel("../data/raw/Telco_customer_churn.xlsx")

In [3]:
df_clean=df.copy()

In [4]:
feature_audit = pd.DataFrame({
    "Column": df_clean.columns,
    "Data Type": df_clean.dtypes.values,
    "Missing Values": df_clean.isnull().sum().values,
    "Unique Values": df_clean.nunique().values
})

feature_audit

,Column,Data Type,Missing Values,Unique Values
0,CustomerID,object,0,7043
1,Count,int64,0,1
2,Country,object,0,1
3,State,object,0,1
4,City,object,0,1129
5,Zip Code,int64,0,1652
6,Lat Long,object,0,1652
7,Latitude,float64,0,1652
8,Longitude,float64,0,1651
9,Gender,object,0,2


In [5]:
feature_audit.to_csv("../reports/feature_audit.csv", index=False)

In [6]:
for col in df_clean.columns:
    if df_clean[col].nunique() == 1:
        print(col)

Count
Country
State


In [7]:
df_clean["CustomerID"].nunique()

7043

In [8]:
leakage_columns = [
    "Churn Label",
    "Churn Value",
    "Churn Score",
    "Churn Reason"
]

df_clean[leakage_columns].head()

,Churn Label,Churn Value,Churn Score,Churn Reason
0,Yes,1,86,Competitor made better offer
1,Yes,1,67,Moved
2,Yes,1,86,Moved
3,Yes,1,84,Moved
4,Yes,1,89,Competitor had better devices


Feature	Decision	Reason
CustomerID	Remove	Unique identifier
Count	?	Verify if constant
Country	?	Verify if constant
Churn Value	Remove	Duplicate encoding of target
Churn Score	Remove	Potential target leakage
Churn Reason	Remove from ML	Post-churn information

In [9]:
for col in df_clean.columns:
    if df_clean[col].nunique() == 1:
        print(col)

Count
Country
State


In [10]:
df_clean["CustomerID"].nunique()

7043

In [11]:
df_clean[["Churn Label","Churn Value","Churn Score","Churn Reason"]].head()

,Churn Label,Churn Value,Churn Score,Churn Reason
0,Yes,1,86,Competitor made better offer
1,Yes,1,67,Moved
2,Yes,1,86,Moved
3,Yes,1,84,Moved
4,Yes,1,89,Competitor had better devices


In [12]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [13]:
columns_to_drop = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "Churn Value",
    "Churn Score",
    "Churn Reason"
]

df_clean = df_clean.drop(columns=columns_to_drop)

In [14]:
df_clean.shape

(7043, 26)

In [15]:
(df_clean == " ").sum()

City                  0
Zip Code              0
Lat Long              0
Latitude              0
Longitude             0
Gender                0
Senior Citizen        0
Partner               0
Dependents            0
Tenure Months         0
Phone Service         0
Multiple Lines        0
Internet Service      0
Online Security       0
Online Backup         0
Device Protection     0
Tech Support          0
Streaming TV          0
Streaming Movies      0
Contract              0
Paperless Billing     0
Payment Method        0
Monthly Charges       0
Total Charges        11
Churn Label           0
CLTV                  0
dtype: int64

In [16]:
df_clean[df_clean["Total Charges"] == " "]

,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,CLTV
2234,San Bernardino,92408,"34.084909, -117.258107",34.084909,-117.258107,Female,No,Yes,No,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No,2578
2438,Independence,93526,"36.869584, -118.189241",36.869584,-118.189241,Male,No,No,No,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No,5504
2568,San Mateo,94401,"37.590421, -122.306467",37.590421,-122.306467,Female,No,Yes,No,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No,2048
2667,Cupertino,95014,"37.306612, -122.080621",37.306612,-122.080621,Male,No,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No,4950
2856,Redcrest,95569,"40.363446, -123.835041",40.363446,-123.835041,Female,No,Yes,No,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No,4740
4331,Los Angeles,90029,"34.089953, -118.294824",34.089953,-118.294824,Male,No,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No,2019
4687,Sun City,92585,"33.739412, -117.173334",33.739412,-117.173334,Male,No,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No,2299
5104,Ben Lomond,95005,"37.078873, -122.090386",37.078873,-122.090386,Female,No,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No,3763
5719,La Verne,91750,"34.144703, -117.770299",34.144703,-117.770299,Male,No,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No,4890
6772,Bell,90201,"33.970343, -118.171368",33.970343,-118.171368,Female,No,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No,2342


In [17]:
df_clean["Total Charges"] = pd.to_numeric(
    df_clean["Total Charges"],
    errors="coerce"
)

In [18]:
df_clean["Total Charges"].isnull().sum()


np.int64(11)

In [19]:
df_clean[df_clean["Total Charges"].isnull()]

,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,CLTV
2234,San Bernardino,92408,"34.084909, -117.258107",34.084909,-117.258107,Female,No,Yes,No,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No,2578
2438,Independence,93526,"36.869584, -118.189241",36.869584,-118.189241,Male,No,No,No,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No,5504
2568,San Mateo,94401,"37.590421, -122.306467",37.590421,-122.306467,Female,No,Yes,No,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No,2048
2667,Cupertino,95014,"37.306612, -122.080621",37.306612,-122.080621,Male,No,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No,4950
2856,Redcrest,95569,"40.363446, -123.835041",40.363446,-123.835041,Female,No,Yes,No,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No,4740
4331,Los Angeles,90029,"34.089953, -118.294824",34.089953,-118.294824,Male,No,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No,2019
4687,Sun City,92585,"33.739412, -117.173334",33.739412,-117.173334,Male,No,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No,2299
5104,Ben Lomond,95005,"37.078873, -122.090386",37.078873,-122.090386,Female,No,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No,3763
5719,La Verne,91750,"34.144703, -117.770299",34.144703,-117.770299,Male,No,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No,4890
6772,Bell,90201,"33.970343, -118.171368",33.970343,-118.171368,Female,No,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No,2342


In [22]:
df_clean[["Tenure Months", "Monthly Charges"]]

,Tenure Months,Monthly Charges
0,2,53.85
1,2,70.70
2,8,99.65
3,28,104.80
4,49,103.70
...,...,...
7038,72,21.15
7039,24,84.80
7040,72,103.20
7041,11,29.60


In [23]:
df_clean.loc[
    df_clean["Total Charges"].isnull(),
    ["Tenure Months", "Monthly Charges", "Total Charges", "Churn Label"]
]

,Tenure Months,Monthly Charges,Total Charges,Churn Label
2234,0,52.55,NaN,No
2438,0,20.25,NaN,No
2568,0,80.85,NaN,No
2667,0,25.75,NaN,No
2856,0,56.05,NaN,No
4331,0,19.85,NaN,No
4687,0,25.35,NaN,No
5104,0,20.00,NaN,No
5719,0,19.70,NaN,No
6772,0,73.35,NaN,No


In [24]:
df_clean["Total Charges"] = df_clean["Total Charges"].fillna(0)

In [25]:
df_clean["Total Charges"].isnull().sum()

np.int64(0)

In [26]:
df_clean["Total Charges"].dtype

dtype('float64')

In [27]:
df_clean.to_csv(
    "../data/processed/telco_customer_churn_clean.csv",
    index=False
)

### Handling Missing Values in `Total Charges`

The `Total Charges` column contained 11 missing values. Investigation revealed that all affected customers had `Tenure Months = 0` and `Churn Label = "No"`, indicating they were newly enrolled customers who had not yet completed a billing cycle.

Since total charges represent the cumulative amount billed over a customer's tenure, a tenure of zero logically results in total charges of zero. Therefore, the missing values were replaced with `0` instead of removing the records or applying statistical imputation. This preserves all customer records while maintaining consistency with the underlying business logic.
 